In [8]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
from sklearn.metrics import classification_report

# Set a fixed seed so results are reproducible every time we run the code
torch.manual_seed(42)

# 1. LOAD DATA
# Read the training and validation CSV files into pandas DataFrames
# Each file has two columns: 'text' (the tweet) and 'label' (the sentiment)
train_df = pd.read_csv("sent_train.csv")
valid_df  = pd.read_csv("sent_valid.csv")

print(f"Train: {len(train_df)} samples  |  Valid: {len(valid_df)} samples")
print(train_df["label"].value_counts())

Train: 9543 samples  |  Valid: 2388 samples
label
2    6178
1    1923
0    1442
Name: count, dtype: int64


In [9]:
# 2. TEXT PREPROCESSING
# We clean the tweets before training to remove noise that does not
# help the model learn sentiment (URLs, stock tickers, hashtags, etc.)
def clean(text):
    text = str(text).lower()               # convert to lowercase
    text = re.sub(r"http\S+", "", text)    # remove URLs
    text = re.sub(r"\$\w+", "", text)      # remove stock tickers like $AAPL
    text = re.sub(r"#\w+", "", text)       # remove hashtags like #Finance
    text = re.sub(r"[^a-z\s]", "", text)   # keep only letters and spaces
    return text.strip()

train_df["clean"] = train_df["text"].apply(clean)
valid_df["clean"]  = valid_df["text"].apply(clean)

In [10]:
# 3. BUILD VOCABULARY
# We create a dictionary that maps each word to a unique integer index.
# We only use words from the training set to avoid data leakage.
counter = Counter()
for text in train_df["clean"]:
    counter.update(text.split())

# Keep only the 10,000 most frequent words to limit memory usage
# <PAD> (index 0) is used for padding shorter sequences
# <UNK> (index 1) replaces words not found in the vocabulary
VOCAB_SIZE = 10_000
vocab = ["<PAD>", "<UNK>"] + [w for w, _ in counter.most_common(VOCAB_SIZE - 2)]
w2i   = {w: i for i, w in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)} tokens")

Vocabulary size: 10000 tokens


In [11]:
# 4. ENCODE SEQUENCES WITH PADDING
# Each tweet is converted into a fixed-length list of integers.
# If a tweet is shorter than MAX_LEN, we pad it with zeros at the end.
# If it is longer, we cut it off at MAX_LEN tokens.
MAX_LEN = 50

def encode(text):
    tokens = text.split()[:MAX_LEN]
    ids = [w2i.get(t, 1) for t in tokens]      # unknown words → index 1
    ids += [0] * (MAX_LEN - len(ids))           # pad with zeros if too short
    return ids

train_df["enc"] = train_df["clean"].apply(encode)
valid_df["enc"]  = valid_df["clean"].apply(encode)

In [12]:
# 5. PYTORCH DATASET AND DATALOADER 
# A Dataset object tells PyTorch how to access individual samples.
# A DataLoader groups samples into batches and shuffles the training data.
class TweetDS(Dataset):
    def __init__(self, df):
        self.X = torch.tensor(df["enc"].tolist(),   dtype=torch.long)
        self.y = torch.tensor(df["label"].tolist(), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(TweetDS(train_df), batch_size=64, shuffle=True)
valid_loader  = DataLoader(TweetDS(valid_df),  batch_size=64)